# Food Recommender System - Data Preprocessing
This notebook covers the data loading, cleaning, and preprocessing steps for the Open Food Facts dataset.

## 1. Import Libraries

In [ ]:
# Import necessary libraries (e.g., PySpark, pandas)
# Example using pandas for now, switch to PySpark if needed
import pandas as pd
import os
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Configure Spark session if using PySpark
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
spark = SparkSession.builder.appName("OFFPreprocessing").getOrCreate()

## 2. Load Data
Load the dataset from the parquet file.

In [ ]:
data_path = '../data/food.parquet'
if os.path.exists(data_path):
    # Using pandas to read parquet
    df = pd.read_parquet(data_path)
    print(f"Data loaded successfully from {data_path}")
    print(f"Shape: {df.shape}")
    display(df.head())
    # If using PySpark:
    # df_spark = spark.read.parquet(data_path)
    # print(f"Data loaded successfully into Spark DataFrame from {data_path}")
    # print(f"Count: {df_spark.count()}")
    # df_spark.printSchema()
    # df_spark.show(5)
else:
    print(f"Error: Data file not found at {data_path}")

## 3. Data Cleaning & Preprocessing
Perform cleaning steps like handling nulls, normalization, tokenization etc.

### 3.1 Select Relevant Columns
Keep only the columns needed for the recommender system, such as product name, categories, ingredients, and nutritional information.

In [ ]:
relevant_columns = [
    'code', 'product_name', 'categories_en', 
    'ingredients_text_en', 'nutriscore_grade', 
    # Add other relevant nutritional columns if needed, e.g.:
    # 'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 
    # 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 
    # 'proteins_100g', 'salt_100g', 'sodium_100g'
]

# Check which relevant columns are actually present in the DataFrame
existing_relevant_columns = [col for col in relevant_columns if col in df.columns]
print(f"Selecting columns: {existing_relevant_columns}")

df_selected = df[existing_relevant_columns].copy()
display(df_selected.head())

### 3.2 Handle Missing Values
Check for and handle missing values in key columns.

In [ ]:
# Check missing values percentage
missing_percentage = df_selected.isnull().sum() * 100 / len(df_selected)
print("Missing value percentage per column:")
print(missing_percentage)

# Decide on a strategy: drop rows with missing essential info (e.g., product_name, ingredients)
# For simplicity, let's drop rows where product_name or ingredients_text_en are missing
essential_cols = ['product_name', 'ingredients_text_en']
df_cleaned = df_selected.dropna(subset=[col for col in essential_cols if col in df_selected.columns]).copy()

print(f"\nShape after dropping rows with missing essential info: {df_cleaned.shape}")

# Fill missing nutriscore_grade with a placeholder like 'unknown' or drop them
if 'nutriscore_grade' in df_cleaned.columns:
    df_cleaned['nutriscore_grade'] = df_cleaned['nutriscore_grade'].fillna('unknown')
    print("Filled missing 'nutriscore_grade' with 'unknown'.")

# Display info after handling missing values
display(df_cleaned.info())
display(df_cleaned.head())

### 3.3 Text Normalization (Example: Lowercasing)
Normalize text fields like product name, categories, and ingredients.

In [ ]:
text_cols = ['product_name', 'categories_en', 'ingredients_text_en']
for col in text_cols:
    if col in df_cleaned.columns:
        # Ensure the column is string type before applying .str methods
        df_cleaned[col] = df_cleaned[col].astype(str).str.lower()
        print(f"Column '{col}' converted to lowercase.")

display(df_cleaned.head())

### 3.4 Further Steps (Placeholder)
Next steps would include:
*   **Tokenization**: Splitting ingredients and categories into individual words/tokens.
*   **Stopword Removal**: Removing common words.
*   **Handling Multi-valued Fields**: Splitting comma-separated categories.
*   **Saving Cleaned Data**: Storing the processed DataFrame.

In [ ]:
# Placeholder for further preprocessing code (Tokenization, Stopwords, etc.)
# Example: Tokenizing ingredients
# if 'ingredients_text_en' in df_cleaned.columns:
#     from nltk.tokenize import word_tokenize # Requires nltk installation
#     df_cleaned['ingredients_tokens'] = df_cleaned['ingredients_text_en'].apply(word_tokenize)
#     display(df_cleaned[['ingredients_text_en', 'ingredients_tokens']].head())

print("Preprocessing steps added. Further implementation needed for tokenization etc.")

In [ ]:
# Tokenize and remove stopwords from ingredients text
if 'ingredients_text_en' in df_cleaned.columns:
    # Initialize stopwords
    stop_words = set(stopwords.words('english'))
    
    # Function to tokenize and remove stopwords
    def tokenize_and_clean(text):
        if pd.isna(text) or text == 'nan':
            return []
        # Tokenize
        tokens = word_tokenize(text.lower())
        # Remove stopwords and non-alphabetic tokens
        tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
        return tokens
    
    # Apply tokenization and stopword removal
    df_cleaned['ingredients_tokens'] = df_cleaned['ingredients_text_en'].apply(tokenize_and_clean)
    print("Tokenized ingredients and removed stopwords.")
    display(df_cleaned[['ingredients_text_en', 'ingredients_tokens']].head())

### 3.5 Handle Multi-valued Fields
Split comma-separated category fields and handle them appropriately.

In [ ]:
# Process categories - split comma-separated values
if 'categories_en' in df_cleaned.columns:
    # Function to split categories and clean them
    def clean_categories(cats_string):
        if pd.isna(cats_string) or cats_string == 'nan':
            return []
        # Split by commas and clean each category
        categories = [cat.strip() for cat in cats_string.split(',')]
        # Remove empty categories
        categories = [cat for cat in categories if cat]
        return categories
    
    df_cleaned['categories_list'] = df_cleaned['categories_en'].apply(clean_categories)
    print("Split categories into lists.")
    display(df_cleaned[['categories_en', 'categories_list']].head())

## 4. Feature Engineering
Convert text and categorical data into numerical features for the recommender system.

### 4.1 Text Vectorization (TF-IDF) for Ingredients

In [ ]:
# Create document strings from ingredient tokens for TF-IDF vectorization
df_cleaned['ingredients_doc'] = df_cleaned['ingredients_tokens'].apply(lambda tokens: ' '.join(tokens))

# Create TF-IDF vectors for ingredients
# Using max_features to limit the vocabulary size and min_df to ignore rare terms
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_cleaned['ingredients_doc'])

print(f"Created TF-IDF matrix for ingredients with shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

# Display sample feature names
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Sample feature names: {list(feature_names[:10])}")

### 4.2 One-Hot Encoding for Categories
Convert categories to one-hot encoding for similarity calculation.

In [ ]:
# Get all unique categories
all_categories = set()
for cat_list in df_cleaned['categories_list']:
    all_categories.update(cat_list)
    
print(f"Total unique categories found: {len(all_categories)}")

# Function to create one-hot encoding for categories
def one_hot_categories(categories, all_cats):
    encoding = {cat: 1 if cat in categories else 0 for cat in all_cats}
    return encoding

# Apply one-hot encoding (only if the number of categories is manageable)
if len(all_categories) < 1000:  # Threshold to decide if one-hot is practical
    df_cleaned['categories_onehot'] = df_cleaned['categories_list'].apply(
        lambda cats: one_hot_categories(cats, all_categories))
    print("Created one-hot encoding for categories.")
    # Display sample
    sample_onehot = df_cleaned['categories_onehot'].iloc[0]
    print(f"Sample one-hot encoding (showing only True values):")
    print({k: v for k, v in sample_onehot.items() if v == 1})
else:
    print("Too many unique categories for one-hot encoding. Consider using embeddings or dimensionality reduction.")

## 5. Computing Similarity Matrices
Compute similarity matrices for ingredients to be used in the recommendation system.

In [ ]:
# Compute cosine similarity on ingredient TF-IDF vectors
# Note: For large datasets, consider using approximate nearest neighbors or computing similarity on-demand

# First, get a smaller sample for demonstration (limit to 1000 products if dataset is large)
if len(df_cleaned) > 1000:
    print("Using a sample of 1000 products for similarity calculation demonstration...")
    sample_indices = np.random.choice(len(df_cleaned), size=1000, replace=False)
    sample_tfidf = tfidf_matrix[sample_indices]
    sample_df = df_cleaned.iloc[sample_indices].copy()
else:
    sample_tfidf = tfidf_matrix
    sample_df = df_cleaned.copy()

# Compute pairwise similarity
cosine_sim = cosine_similarity(sample_tfidf, sample_tfidf)

print(f"Computed cosine similarity matrix with shape: {cosine_sim.shape}")

# Display example similarities for first product
first_product_name = sample_df['product_name'].iloc[0]
print(f"\nTop 5 similar products to '{first_product_name}':")

# Get the indices of the 5 most similar products (excluding itself)
similar_indices = cosine_sim[0].argsort()[-6:-1][::-1]
for idx in similar_indices:
    similarity_score = cosine_sim[0, idx]
    similar_product = sample_df['product_name'].iloc[idx]
    print(f"  - {similar_product} (Similarity: {similarity_score:.3f})")

## 6. Save Processed Data
Save the processed dataframe and feature matrices for use in the recommendation system.

In [ ]:
import pickle

# Save the cleaned dataframe
cleaned_data_path = '../data/cleaned_food_data.parquet'
df_cleaned.to_parquet(cleaned_data_path)
print(f"Saved cleaned data to: {cleaned_data_path}")

# Save the TF-IDF vectorizer and similarity matrix (for the recommendation system)
with open('../data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("Saved TF-IDF vectorizer.")

# If using a sample, save the full similarity matrix (careful with large datasets)
if len(df_cleaned) <= 1000:
    with open('../data/cosine_sim_matrix.pkl', 'wb') as f:
        pickle.dump(cosine_sim, f)
    print("Saved full cosine similarity matrix.")
else:
    print("Dataset too large - similarity will be computed on-demand in the recommender system.")
    
print("\nPreprocessing complete! Data is ready for the recommendation system.")